In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_key = user_secrets.get_secret("wandb-key")


In [3]:
import wandb
wandb.login(key = WB_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import re
import string
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    TfidfVectorizer
)
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

OPTION_LABELS = ["A", "B", "C", "D", "E"]
TEXT_COLUMNS = ["prompt", "A", "B", "C", "D", "E"]
ANSWER_COLUMN = "answer"

In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv") 
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
ss = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train.shape

(2000, 8)

In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [ ]:
train.describe(include ="all").T

In [ ]:
missing_values = train.isnull().sum()
missing_values

In [ ]:
answer_distribution = (
    train["answer"]
    .value_counts()
    .sort_index()
)

print(answer_distribution)

answer_distribution.plot(
    kind="bar",
    title="Answer Distribution"
)

In [ ]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = text.lower()

    text = text.replace("\n", " ")

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
for column in TEXT_COLUMNS:

    train[f"{column}_clean"] = (
        train[column]
        .apply(clean_text)
    )

In [ ]:
train["prompt_tokens"] = (
    train["prompt_clean"]
    .str.split()
)

In [ ]:
vocab = set()

for tokens in train["prompt_tokens"]:
    vocab.update(tokens)

print(
    "Vocabulary Size:",
    len(vocab)
)

In [ ]:
sample_tokens = train.iloc[0]["prompt_tokens"]

filtered = [

    word

    for word in sample_tokens

    if word not in ENGLISH_STOP_WORDS

]

print(filtered)

In [ ]:
corpus = []

for col in [
    "prompt_clean",
    "A_clean",
    "B_clean",
    "C_clean",
    "D_clean",
    "E_clean"
]:

    corpus.extend(
        train[col].tolist()
    )

vectorizer = TfidfVectorizer(
    ngram_range=(1,2)
)

vectorizer.fit(corpus)

print(
    len(vectorizer.vocabulary_)
)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

row = train.iloc[0]

prompt_vec = vectorizer.transform(
    [row["prompt_clean"]]
)

option_vecs = vectorizer.transform([

    row["A_clean"],
    row["B_clean"],
    row["C_clean"],
    row["D_clean"],
    row["E_clean"]

])

scores = cosine_similarity(
    prompt_vec,
    option_vecs
)[0]

scores

In [ ]:
OPTION_LABELS = [
    "A",
    "B",
    "C",
    "D",
    "E"
]

predictions = []

for i, row in train.iterrows():

    prompt_vec = vectorizer.transform(
        [row["prompt_clean"]]
    )

    option_vecs = vectorizer.transform([

        row["A_clean"],
        row["B_clean"],
        row["C_clean"],
        row["D_clean"],
        row["E_clean"]

    ])

    sims = cosine_similarity(
        prompt_vec,
        option_vecs
    )[0]

    ranked = np.argsort(sims)[::-1]

    predictions.append(

        [

            OPTION_LABELS[i]

            for i in ranked[:3]

        ]

    )

In [ ]:
def apk(actual, predicted):

    predicted = predicted[:3]

    if actual in predicted:

        return 1 / (
            predicted.index(actual)+1
        )

    return 0


def mapk(actuals, predictions):

    return np.mean(

        [

            apk(a,p)

            for a,p

            in zip(
                actuals,
                predictions
            )

        ]

    )

In [ ]:
tfidf_score = mapk(
    train["answer"],
    predictions
)

print(tfidf_score)

In [ ]:
from gensim.models import Word2Vec
corpus = []

for col in [
    "prompt_clean",
    "A_clean",
    "B_clean",
    "C_clean",
    "D_clean",
    "E_clean"
]:
    corpus.extend(train[col].str.split().tolist())

word2vec_model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)


In [ ]:
word2vec_predictions = []

for i, row in train.iterrows():

    prompt_tokens = row["prompt_clean"].split()

    prompt_vectors = [
        word2vec_model.wv[word]
        for word in prompt_tokens
        if word in word2vec_model.wv
    ]

    if len(prompt_vectors) == 0:
        prompt_embedding = np.zeros(word2vec_model.vector_size)
    else:
        prompt_embedding = np.mean(prompt_vectors, axis=0)

    similarities = []

    for option in ["A", "B", "C", "D", "E"]:

        option_tokens = row[f"{option}_clean"].split()

        option_vectors = [
            word2vec_model.wv[word]
            for word in option_tokens
            if word in word2vec_model.wv
        ]

        if len(option_vectors) == 0:
            option_embedding = np.zeros(word2vec_model.vector_size)
        else:
            option_embedding = np.mean(option_vectors, axis=0)

        if (
            np.linalg.norm(prompt_embedding) == 0
            or np.linalg.norm(option_embedding) == 0
        ):
            similarity = 0
        else:
            similarity = cosine_similarity(
                prompt_embedding.reshape(1, -1),
                option_embedding.reshape(1, -1)
            )[0][0]

        similarities.append((option, similarity))

    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top3 = [option for option, _ in similarities[:3]]

    word2vec_predictions.append(top3)

In [ ]:
word2vec_score = mapk(
    train["answer"],
    word2vec_predictions
)

print(f"Word2Vec MAP@3: {word2vec_score:.5f}")

In [ ]:
results = pd.DataFrame({

    "Model":[

        "TF-IDF",
        "Word2Vec"

    ],

    "MAP@3":[

        tfidf_score,
        word2vec_score

    ]

})

results

In [ ]:
errors = train[

    [

        actual not in pred

        for actual,pred

        in zip(

            train["answer"],
            predictions

        )

    ]

]

errors.head()

# Milestone -02

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, pipeline

import torch

In [ ]:
train["combined_text"] = (
    "Question:" + train["prompt"].astype(str)+
    "[SEP] A:" + train["A"].astype(str)+
    "[SEP] B: " + train["B"].astype(str) +
    "[SEP] C: " + train["C"].astype(str) +
    "[SEP] D: " + train["D"].astype(str) +
    "[SEP] E: " + train["E"].astype(str)
)

train["combined_text"].head()

In [ ]:
hf_dataset = Dataset.from_pandas(train)
print(hf_dataset)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_question(q_rec):
    return tokenizer(
        q_rec["combined_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_data = hf_dataset.map(tokenize_question)

In [ ]:
print(type(tokenizer))

In [ ]:
bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
sample_text = train.loc[0, "combined_text"]

encoded_input = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation = True,
    padding = True,
    max_length = 128
)

with torch.no_grad():
    model_output = bert_model(**encoded_input)

In [ ]:
print(model_output.last_hidden_state.shape)

In [ ]:
cls_embd = model_output.last_hidden_state[:, 0, :]
print(cls_embd.shape)

In [ ]:
print(cls_embd)

In [ ]:
zero_shot_classifier= pipeline(
    "zero-shot-classification",
    model = "facebook/bart-large-mnli"
)

In [ ]:
sample_ques = train.loc[0, "prompt"]
print(sample_ques)

In [ ]:
candidate_lbls = [
    "Science",
    "Mathematics",
    "History",
    "Geography",
    "Technology"
]

In [ ]:
pred= zero_shot_classifier(
    sample_ques,
    candidate_lbls
)
pred

In [ ]:
print("Predicted Labels:")
print(pred["labels"])

print("\nConfidence Scores:")
print(pred["scores"])

In [ ]:
print("Most Likely Category:", pred["labels"][0])
print("Confidence Score:", round(pred["scores"][0], 4))

# milestone -03

In [9]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.8 MB/s eta 0:00:00:00:0100:01


In [10]:
import faiss
from sentence_transformers import SentenceTransformer

In [22]:
knowledge_base = []

for _, row in train.iterrows():

    document = (
        f"Question: {row['prompt']}\n"
        f"A: {row['A']}\n"
        f"B: {row['B']}\n"
        f"C: {row['C']}\n"
        f"D: {row['D']}\n"
        f"E: {row['E']}"
    )

    knowledge_base.append(document)

print("Total Documents:", len(knowledge_base))

print("\nFirst Knowledge Document:\n")
print(knowledge_base[0])

Total Documents: 2000

First Knowledge Document:

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present

In [23]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.


In [24]:
document_embeddings = embedding_model.encode(
    knowledge_base,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(document_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


In [25]:
embedding_dimension = document_embeddings.shape[1]

vector_database = faiss.IndexFlatL2(
    embedding_dimension
)

print(vector_database)

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x78daf02793f0> >


In [26]:
vector_database.add(document_embeddings)
print("Total Stored Questions:", vector_database.ntotal)

Total Stored Questions: 2000


In [27]:
query_question = train.loc[0, "prompt"]

print("Query Question:")
print(query_question)

query_embedding = embedding_model.encode(
    [query_question],
    convert_to_numpy=True
)

Query Question:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [28]:

top_k = 5

distances, indices = vector_database.search(
    query_embedding,
    top_k
)

In [29]:

print("Retrieved Questions:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"{rank}. {knowledge_base[idx]}")

Retrieved Questions:

1. Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.
D: Martin Heidegger bel

In [30]:
retrieved_context = "\n".join(
    [knowledge_base[idx] for idx in indices[0]]
)

augmented_prompt = f"""
Context:
{retrieved_context}

Question:
{train.loc[0, 'prompt']}

Options:
A. {train.loc[0, 'A']}
B. {train.loc[0, 'B']}
C. {train.loc[0, 'C']}
D. {train.loc[0, 'D']}
E. {train.loc[0, 'E']}
"""

print(augmented_prompt)


Context:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.
D: Martin Heidegger believes that the 